In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Unzip the compressed IMDb dataset (retrieved from https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) stored in Google Drive to the same folder
!unzip "/content/drive/MyDrive/MSc Thesis/IMDb Dataset.csv.zip" -d "/content/drive/MyDrive/MSc Thesis/"

Archive:  /content/drive/MyDrive/MSc Thesis/IMDb Dataset.csv.zip
  inflating: /content/drive/MyDrive/MSc Thesis/IMDB Dataset.csv  


In [ ]:
# Install required packages
!pip install -q transformers tqdm

In [ ]:
# Import required libraries
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
from tqdm import tqdm
import torch
import os

In [ ]:
# Load the MADLAD-400-3B-MT translation model and the tokenizer from Hugging Face (https://huggingface.co/google/madlad400-3b-mt)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/madlad400-3b-mt",
    dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/madlad400-3b-mt")
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/742 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

spiece.model:   0%|          | 0.00/4.43M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(256000, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(256000, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=2048, bias=False)
              (k): Linear(in_features=1024, out_features=2048, bias=False)
              (v): Linear(in_features=1024, out_features=2048, bias=False)
              (o): Linear(in_features=2048, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=8192, bias=False)
              (wi_1): Linear(in_features=1024, out_features=8192, bias=False)
     

In [ ]:
# Load the IMDb dataset from Google Drive
data = pd.read_csv("/content/drive/MyDrive/MSc Thesis/IMDb Dataset.csv")
# Print the first few and the last few rows of the dataset
print(data.head())
print("\n")
print(data.tail())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


                                                  review sentiment
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative


In [ ]:
# Split the IMDb dataset into training and test sets (50/50 split)
# First 25000 rows compose the training set
training_df = data.iloc[:25000]
# Remaining rows (from 25000 onward) compose the test set
test_df = data.iloc[25000:]
# Print the first few and the last few rows of the training set
print("Training set:")
print(training_df.head())
print("\n")
print(training_df.tail())
print("\n")
# Print the first few and the last few rows of the test set
print("Test set:")
print(test_df.head())
print("\n")
print(test_df.tail())

Training set:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


                                                  review sentiment
24995  This movie was a real torture fest to sit thro...  negative
24996  John Wayne & Albert Dekker compete for oil rig...  negative
24997  Tarantino once remarked on a melodrama from th...  positive
24998  Aah yes the workout show was a great. Not only...  positive
24999  This film should have never been made. Honestl...  negative


Test set:
                                                  review sentiment
25000  This movie was bad from the start. The only pu...  negative
25001  God, I never felt so insulted in my whole life...  

In [ ]:
# Sample 100 positive and 100 negative reviews from the test set to create a balanced test set
test_sample = (test_df.groupby("sentiment", group_keys=False).sample(n=100, random_state=42).sample(frac=1, random_state=42).reset_index(drop=True))
# Save the balanced test set
output_path = "/content/drive/MyDrive/MSc Thesis/sample_test_data.csv"
test_sample.to_csv(output_path, index=False)
print(f"Saved test set file to {output_path}")

Saved test set file to /content/drive/MyDrive/MSc Thesis/sample_test_data.csv


In [ ]:
# Translation function
def translate_texts(texts, desc, batch_size=2):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    translated = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = [f"<2el> {text}" for text in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translated.extend(decoded)
    return translated

# Sampling and translation function with checkpointing to create balanced training sets
def sample_and_translate(set_sizes=[1000, 2000]):
    for set_size in set_sizes:
        output_path = f"/content/drive/MyDrive/MSc Thesis/sample_training_data_{set_size}.csv"
        # Checkpoint: Skip translation, if the file already exists
        if os.path.exists(output_path):
            print(f"{output_path} already exists. Skipping...")
            continue
        # Sample an equal number of positive and negative reviews from the training set to create a balanced training set
        training_sample = (training_df.groupby("sentiment", group_keys=False).sample(n=set_size // 2, random_state=42).sample(frac=1, random_state=42).reset_index(drop=True))
        print(f"Translating {set_size} samples...")
        # Translate the reviews to Greek
        training_sample["review"] = translate_texts(training_sample["review"], desc=f"Translating {set_size} reviews")
        # Map the sentiment labels to Greek
        label_map = {"positive": "θετικό", "negative": "αρνητικό"}
        training_sample["sentiment"] = training_sample["sentiment"].map(label_map)
        # Save the balanced and translated training set
        training_sample.to_csv(output_path, index=False)
        print(f"Saved translated training set file to {output_path}")

# Run the sampling and translation function
sample_and_translate()

/content/drive/MyDrive/MSc Thesis/sample_training_data_1000.csv already exists. Skipping...
/content/drive/MyDrive/MSc Thesis/sample_training_data_2000.csv already exists. Skipping...
